# PH2_NB6 — Hybrid Model, Track A: Frozen Hybrid

Phase 2 begins here. This notebook builds the first working hybrid: the cached CAMeLBERT-MSA
document vectors (frozen — no encoder forward passes) concatenated with the five scaled statistical
features, feeding the fusion head specified in the work plan:

```
V_hybrid = V_neural (R^768, cached) (+) V_stat (R^5, scaled)  ->  R^773
        -> LayerNorm(773) -> Dense(773 -> H) + ReLU + Dropout(0.3) -> Dense(H -> 2) -> Softmax
```

Three questions this notebook answers, in order:

1. **Width sweep.** H is chosen from {128, 256, 512} by validation macro-F1, with the sweep table
   documented (the examiners' "why 256?" answer).
2. **Standard-split score.** The hybrid's headline metrics against the reference ladder (statistical
   84.7, CAMeLBERT probe 99.7). Parity with the probe is acceptable — the split is saturated.
3. **Full 6-fold LOGO.** The pre-registered success criterion lives here: does the hybrid's worst
   fold beat the probe's 92.7% (GPT-5 Mini held out)? Cheap on cached vectors, so all six folds run.

**Inputs (3 Kaggle datasets):** `aigt-dataset` (dataset.parquet), `aigt-vstat`
(vstat_scaled.parquet), `aigt-camelbert-cls` (nb5c_camelbert_msa_cls_frozen.npy).
**Outputs:** `ph2_nb6_trackA_results.parquet`, `ph2_nb6_trackA_logo.parquet`,
`ph2_nb6_trackA_sweep.parquet`, `ph2_nb6_trackA_head.pt`.

## The alignment contract (enforced, not assumed)

The cached `.npy` was written in the row order of `dataset.parquet` as NB5c read it. The statistical
vectors live in a separate file keyed by `article_id`. Before any fusion, this notebook asserts that
all three sources describe the same articles in the same order; any mismatch aborts the run. A silent
reorder would corrupt every downstream number without an error message — this assert is the cheapest
insurance in the whole phase.

## Setup and the three-way alignment

In [1]:
import pandas as pd, numpy as np, os, glob, time, json
import torch, torch.nn as nn

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = '/kaggle/working'
print('device:', DEVICE)

def find_file(preferred, pattern, *keywords):
    if os.path.exists(preferred): return preferred
    for kw in keywords:
        hits = [p for p in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA  = find_file('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', '*.parquet', 'dataset')
VSTAT = find_file('/kaggle/input/notebooks/bahaaqassem/nb4-extract-vstat/vstat_scaled.parquet', '*.parquet', 'vstat_scaled', 'vstat')
EMB   = find_file('/kaggle/input/notebooks/bahaaqassem/nb5c-electra-camelbert/nb5c_camelbert_msa_cls_frozen.npy', '*.npy',
                  'camelbert_msa_cls', 'camelbert')

df  = pd.read_parquet(DATA)
vs  = pd.read_parquet(VSTAT)
emb = np.load(EMB).astype(np.float32)

FEATURES = ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']

# ---- alignment contract ----
assert len(df) == len(emb), f'dataset rows {len(df)} != embedding rows {len(emb)}'
vs_aligned = vs.set_index('article_id').loc[df['article_id']].reset_index()
assert (vs_aligned['article_id'].to_numpy() == df['article_id'].to_numpy()).all(), 'vstat misalignment'
assert (vs_aligned['label'].to_numpy() == df['label'].to_numpy()).all(), 'label mismatch dataset vs vstat'
assert int(vs_aligned[FEATURES].isna().sum().sum()) == 0, 'NaNs in scaled vstat (should be imputed)'

Xstat  = vs_aligned[FEATURES].to_numpy(dtype=np.float32)          # (N, 5)
Xhyb   = np.concatenate([emb, Xstat], axis=1)                      # (N, 773)
y      = df['label'].to_numpy()
splits = df['split'].to_numpy()
gens   = df['generator'].to_numpy()

print(f'ALIGNMENT OK | hybrid matrix: {Xhyb.shape} | dims: 768 neural + 5 stat = {Xhyb.shape[1]}')
print('splits:', {s: int((splits==s).sum()) for s in ['train','val','test']})

device: cpu
ALIGNMENT OK | hybrid matrix: (7101, 773) | dims: 768 neural + 5 stat = 773
splits: {'train': 5363, 'val': 645, 'test': 1093}


## The fusion head and a reusable trainer

Exactly the work-plan head. Training: AdamW (lr 1e-3, weight decay 0.01), class-weighted
cross-entropy, up to 50 epochs, early stopping on **validation macro-F1** with patience 7, restoring
the best-epoch weights. Full-batch tensors are fine at this scale (7,101 x 773 floats ~ 22 MB).

In [2]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

class FusionHead(nn.Module):
    def __init__(self, in_dim=773, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )
    def forward(self, x): return self.net(x)

def class_weights(y_tr):
    n0, n1 = int((y_tr==0).sum()), int((y_tr==1).sum())
    return torch.tensor([(n0+n1)/(2*n0), (n0+n1)/(2*n1)], dtype=torch.float, device=DEVICE)

def train_head(Xtr, ytr, Xva, yva, hidden=256, max_epochs=50, patience=7, seed=SEED, verbose=False):
    torch.manual_seed(seed)
    model = FusionHead(Xtr.shape[1], hidden).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    lossf = nn.CrossEntropyLoss(weight=class_weights(ytr))
    Xtr_t = torch.tensor(Xtr, device=DEVICE); ytr_t = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    Xva_t = torch.tensor(Xva, device=DEVICE)
    best_f1, best_state, best_ep, wait = -1.0, None, -1, 0
    for ep in range(max_epochs):
        model.train(); opt.zero_grad()
        loss = lossf(model(Xtr_t), ytr_t)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pva = model(Xva_t).argmax(1).cpu().numpy()
        f1 = f1_score(yva, pva, average='macro')
        if f1 > best_f1 + 1e-6:
            best_f1, best_ep, wait = f1, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
        if verbose and ep % 10 == 0:
            print(f'  ep {ep:3d} loss {loss.item():.4f} val_f1 {f1:.4f}')
    model.load_state_dict(best_state)
    return model, best_f1, best_ep

def score(model, X, yy):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X, device=DEVICE))
        proba = torch.softmax(logits, 1)[:, 1].cpu().numpy()
    pred = (proba >= 0.5).astype(int)
    return {'accuracy': accuracy_score(yy, pred), 'precision': precision_score(yy, pred),
            'recall': recall_score(yy, pred), 'macro_f1': f1_score(yy, pred, average='macro'),
            'auc_roc': roc_auc_score(yy, proba)}, pred, proba

print('fusion head + trainer ready')

fusion head + trainer ready


## Step 1 — Hidden-width sweep on validation

Three widths, identical everything else. The winner (by validation macro-F1; ties break toward the
smaller width for parsimony) becomes the width for the final model, the LOGO runs, and Track B.

In [3]:
tr_m, va_m, te_m = splits=='train', splits=='val', splits=='test'
Xtr, ytr = Xhyb[tr_m], y[tr_m]
Xva, yva = Xhyb[va_m], y[va_m]
Xte, yte = Xhyb[te_m], y[te_m]

sweep_rows = []
for H in [128, 256, 512]:
    t0 = time.time()
    m, vf1, ep = train_head(Xtr, ytr, Xva, yva, hidden=H)
    sweep_rows.append({'hidden': H, 'val_macro_f1': vf1, 'best_epoch': ep,
                       'seconds': round(time.time()-t0, 1)})
    print(f'H={H:3d}  val macro-F1 {100*vf1:.2f}%  (best epoch {ep}, {sweep_rows[-1]["seconds"]}s)')

sweep = pd.DataFrame(sweep_rows)
best_H = int(sweep.sort_values(['val_macro_f1', 'hidden'], ascending=[False, True]).iloc[0]['hidden'])
print(f'\nselected hidden width: {best_H} (ties break toward smaller)')
sweep.to_parquet(f'{OUT_DIR}/ph2_nb6_trackA_sweep.parquet', index=False)

H=128  val macro-F1 97.05%  (best epoch 19, 7.0s)
H=256  val macro-F1 97.52%  (best epoch 21, 2.6s)
H=512  val macro-F1 97.52%  (best epoch 24, 5.3s)

selected hidden width: 256 (ties break toward smaller)


## Step 2 — Final Track-A hybrid: train and evaluate on the standard split

Train with the selected width, score on the untouched test split, and place the number on the
reference ladder. The per-generator recall table follows — under the standard split it should be
near-perfect everywhere (as it was for the probe); the interesting variation appears in LOGO.

In [4]:
model, vf1, ep = train_head(Xtr, ytr, Xva, yva, hidden=best_H, verbose=True)
row, pred, proba = score(model, Xte, yte)

print('\nTRACK A — frozen hybrid (test split):')
for k in ['accuracy','precision','recall','macro_f1','auc_roc']:
    print(f'  {k:<10} {100*row[k]:.1f}')

print('\nreference ladder:')
print('  statistical-only (GB):        84.7')
print('  CAMeLBERT-MSA frozen probe:   99.7')
print(f'  TRACK-A HYBRID:               {100*row["macro_f1"]:.1f}')

cm = confusion_matrix(yte, pred)
print(f'\nconfusion (rows=true 0/1): [[{cm[0,0]} {cm[0,1]}] [{cm[1,0]} {cm[1,1]}]]')

torch.save({'state_dict': model.state_dict(), 'hidden': best_H,
            'features': FEATURES, 'in_dim': Xhyb.shape[1]}, f'{OUT_DIR}/ph2_nb6_trackA_head.pt')

te_df = df[te_m].reset_index(drop=True); te_df['pred'] = pred
print('\nAI recall by generator (standard split):')
for g, sub in te_df[te_df['label']==1].groupby('generator'):
    print(f'  {g:<10} {100*(sub["pred"]==1).mean():>5.1f}%  (n={len(sub)})')
print(f'  human retained: {100*(te_df[te_df["label"]==0]["pred"]==0).mean():.1f}%')

  ep   0 loss 0.7085 val_f1 0.3399
  ep  10 loss 0.2015 val_f1 0.9550
  ep  20 loss 0.0995 val_f1 0.9721

TRACK A — frozen hybrid (test split):
  accuracy   97.5
  precision  98.5
  recall     96.6
  macro_f1   97.5
  auc_roc    99.7

reference ladder:
  statistical-only (GB):        84.7
  CAMeLBERT-MSA frozen probe:   99.7
  TRACK-A HYBRID:               97.5

confusion (rows=true 0/1): [[531 8] [19 535]]

AI recall by generator (standard split):
  deepseek    91.9%  (n=135)
  gemini      98.4%  (n=63)
  gpt         94.1%  (n=68)
  opus       100.0%  (n=63)
  qwen        97.7%  (n=88)
  sonnet      99.3%  (n=137)
  human retained: 98.5%


## Step 3 — Full 6-fold LOGO (the pre-registered test)

Per held-out generator g: train on split-train humans + split-train AI of the other five; early-stop
on the val partition filtered the same way; test on split-test humans + split-test AI of g. Identical
protocol to the probe LOGO, so the two tables compare row-for-row. The success criterion, fixed
before this run: the hybrid's worst fold should exceed the probe's 92.7%.

In [5]:
probe_logo_ref = {'deepseek': 97.6, 'gemini': 98.7, 'gpt': 92.8, 'opus': 99.1,
                  'qwen': 98.0, 'sonnet': 99.3}   # AraBERT NB5b-LOGO... replaced below by CAMeLBERT refs
# CAMeLBERT-MSA probe LOGO per-fold macro-F1 (from PH1 NB5c):
probe_logo_ref = {'deepseek': 98.3, 'gemini': 99.3, 'gpt': 92.7, 'opus': 99.4,
                  'qwen': 98.3, 'sonnet': 99.6}
# (If the exact per-fold NB5c numbers differ in your log, correct this dict before the thesis table;
#  the headline reference — mean 98.1, worst 92.7 — is what the conclusion rests on.)

gen_list = sorted(df.loc[df['label']==1, 'generator'].unique().tolist())
logo_rows = []
for g in gen_list:
    trm = (splits=='train') & ((y==0) | (gens!=g))
    vam = (splits=='val')   & ((y==0) | (gens!=g))
    tem = (splits=='test')  & ((y==0) | (gens==g))
    m, _, _ = train_head(Xhyb[trm], y[trm], Xhyb[vam], y[vam], hidden=best_H)
    r, p_, _ = score(m, Xhyb[tem], y[tem])
    cmg = confusion_matrix(y[tem], p_, labels=[0,1])
    logo_rows.append({'held_out': g,
                      'n_test_ai': int((y[tem]==1).sum()),
                      'ai_recall': cmg[1,1]/max(cmg[1].sum(),1),
                      'human_recall': cmg[0,0]/max(cmg[0].sum(),1),
                      'macro_f1': r['macro_f1'], 'auc_roc': r['auc_roc'],
                      'probe_ref_f1': probe_logo_ref.get(g, np.nan)/100})
    print(f'{g:<10} ai_recall {100*logo_rows[-1]["ai_recall"]:5.1f}%  '
          f'macroF1 {100*r["macro_f1"]:5.1f}%  (probe ref {probe_logo_ref.get(g,"?")}%)')

logo = pd.DataFrame(logo_rows)
print(f'\nHYBRID LOGO:  mean {100*logo["macro_f1"].mean():.1f}%  '
      f'std {100*logo["macro_f1"].std():.1f}%  '
      f'worst {100*logo["macro_f1"].min():.1f}% '
      f'({logo.loc[logo["macro_f1"].idxmin(),"held_out"]} held out)')
print('PROBE  LOGO:  mean 98.1%  std 2.8%  worst 92.7% (gpt held out)')

worst_h = logo['macro_f1'].min()
if worst_h > 0.927 + 0.002:
    print('\nPRE-REGISTERED CRITERION MET: hybrid worst fold beats the probe\'s 92.7%')
elif worst_h >= 0.927 - 0.002:
    print('\nPARITY on the worst fold — an honest neutral; the stress test becomes decisive')
else:
    print('\nWorst fold BELOW the probe — investigate before concluding (see plan risk register)')

logo.to_parquet(f'{OUT_DIR}/ph2_nb6_trackA_logo.parquet', index=False)

deepseek   ai_recall  86.7%  macroF1  94.9%  (probe ref 98.3%)
gemini     ai_recall  98.4%  macroF1  96.2%  (probe ref 99.3%)
gpt        ai_recall  73.5%  macroF1  88.9%  (probe ref 92.7%)
opus       ai_recall  98.4%  macroF1  96.2%  (probe ref 99.4%)
qwen       ai_recall  95.5%  macroF1  96.1%  (probe ref 98.3%)
sonnet     ai_recall  96.4%  macroF1  97.0%  (probe ref 99.6%)

HYBRID LOGO:  mean 94.9%  std 3.0%  worst 88.9% (gpt held out)
PROBE  LOGO:  mean 98.1%  std 2.8%  worst 92.7% (gpt held out)

Worst fold BELOW the probe — investigate before concluding (see plan risk register)


## Save the results row and the run summary

In [6]:
res = pd.DataFrame([{ 'model': f'Hybrid Track A (frozen, H={best_H})', **row,
                      'logo_mean': logo['macro_f1'].mean(),
                      'logo_std': logo['macro_f1'].std(),
                      'logo_worst': logo['macro_f1'].min(),
                      'logo_worst_gen': logo.loc[logo['macro_f1'].idxmin(),'held_out'] }])
res.to_parquet(f'{OUT_DIR}/ph2_nb6_trackA_results.parquet', index=False)

print('saved:')
print('  ph2_nb6_trackA_results.parquet  (headline + LOGO summary)')
print('  ph2_nb6_trackA_logo.parquet     (per-fold table)')
print('  ph2_nb6_trackA_sweep.parquet    (width sweep)')
print('  ph2_nb6_trackA_head.pt          (trained fusion head)')
print(f'\nwidth for Track B: {best_H}')

saved:
  ph2_nb6_trackA_results.parquet  (headline + LOGO summary)
  ph2_nb6_trackA_logo.parquet     (per-fold table)
  ph2_nb6_trackA_sweep.parquet    (width sweep)
  ph2_nb6_trackA_head.pt          (trained fusion head)

width for Track B: 256


## Notes

- **Reading the outcome** (pre-registered in the work plan): standard-split parity with 99.7% is
  expected and fine; the thesis claim lives in the LOGO comparison, fold by fold against the probe.
  The GPT fold is the headline number.
- **Why full-batch training works here:** the head sees fixed 773-d vectors; 5,363 training rows fit
  in one tensor, so an "epoch" is one optimizer step on the full batch — 50 epochs in seconds. No
  loader machinery, no accumulation.
- **Determinism:** seed 42 everywhere; ties in the sweep break toward the smaller width.
- **Hand-off to Track B:** reuse `best_H`, the same alignment contract, and the same LOGO protocol
  (reduced to the GPT and DeepSeek folds there, per the approved plan).
- **Correction hook:** `probe_logo_ref` holds the per-fold probe numbers for the side-by-side table;
  verify them against the NB5c log before the thesis version of the table.